In [13]:
# Notebook to define and evaluate a Siamese UNet with a custom weighting schema 

In [14]:
import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("CWD:", os.getcwd())
print("Project root added:", project_root)

CWD: c:\Dev\Damage_Assessment_on_xBD\Model_Architectures Notebooks
Project root added: c:\Dev\Damage_Assessment_on_xBD


In [15]:
import torch
import torch.optim as optim

from src.dataloader import get_loaders
from src.eval import test_evaluation
from src.train import ComboLoss, run_training, plot_train_history
from  src.model_siamese import SiameseUNet
from torch.amp import GradScaler

scaler = GradScaler('cuda')

In [16]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: NVIDIA GeForce RTX 4070


In [17]:
# Config Hyperparameters (Adjust)
NUM_CLASSES = 5
BATCH_SIZE = 4
LR = 1e-4
NUM_EPOCHS = 50
PATIENCE = 15
BASE_FEATURES = 24
SAVE_PATH = "best_siamese_unet_combo.pth"

In [18]:
# Create Data Splits
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from Preprocessing.xBD_splits import create_splits
img_dir = r"..\Preprocessing\tiles\images"
mask_dir = r"..\Preprocessing\tiles\masks"
train_files, val_files, test_files = create_splits(img_dir, mask_dir)
print(f"Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}")

Train: 9820
Val: 1960
Test: 1912
Train: 9820  Val: 1960  Test: 1912


In [19]:
train_loader, val_loader, test_loader = get_loaders(
    train_files, val_files, test_files, img_dir, mask_dir,
    batch_size=BATCH_SIZE, num_workers = 4, pin_memory = True, siamese=True, augment_train=True)

In [20]:
# Sanity check
pre, post, masks = next(iter(train_loader))

print(f"Pre image:  {pre.shape}")    # [B, 3, H, W]
print(f"Post image: {post.shape}")   # [B, 3, H, W]
print(f"Masks:      {masks.shape}")  # [B, H, W]
print(f"Mask range: {masks.min()} – {masks.max()}")
print(f"Device:     {pre.device}")

Pre image:  torch.Size([4, 3, 512, 512])
Post image: torch.Size([4, 3, 512, 512])
Masks:      torch.Size([4, 512, 512])
Mask range: 0 – 4
Device:     cpu


In [21]:
# Loss — ComboLoss (CE + Tversky)
"""class_counts = torch.tensor([
    3258249517, 287893139, 15076019, 19138232, 8918741
], dtype=torch.float32)

freqs = class_counts / class_counts.sum()
weights = 1.0 / torch.sqrt(freqs)
weights = weights / weights.mean()
class_weights = weights.to(device)
"""
class_weights = torch.tensor([0.5, 2.0, 20.0, 18.0, 4.0], dtype=torch.float32).to(device)

In [22]:
# Model
model = SiameseUNet(
    num_classes=NUM_CLASSES,
    base_features=BASE_FEATURES,
    in_channels=3 # only include 3 channels for Siamese
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(next(model.parameters()).device)

Model parameters: 5,839,277
cuda:0


In [23]:
# Loss, Optimizer, LR Scheduler (Define Here)
criterion = ComboLoss(class_weights=class_weights,
                      ce_weight=0.3,
                      tversky_weight=0.7, tversky_alpha=0.85, tversky_beta=0.15, tversky_gamma=0.6,
                      num_classes=NUM_CLASSES).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay = 1e-4)

#scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=2e-4, steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS, pct_start=0.1, div_factor=10, final_div_factor=100)

In [24]:
history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    num_classes=NUM_CLASSES,
    save_path=SAVE_PATH, verbose = True,
    siamese = True, scaler=scaler)



Epoch 1/50  (lr=2.00e-05)
  Train Loss: 1.0525  |  Val Loss: 0.8854
  Train mIoU: 0.1993  |  Val mIoU: 0.2237
  Train Acc:  0.6642  |  Val Acc:  0.8641
  Val Per-Class:
    Class                IoU     Prec   Recall
    -----------------------------------------
    No Damage         0.8784   0.9914   0.8851
    Minor             0.2337   0.3188   0.4667
    Major             0.0008   0.0008   0.4709
    Destroyed         0.0058   0.0059   0.4143
    Unclassified      0.0000   0.0056   0.0000
  *** Saved new best model (mIoU=0.2237) ***


  *** Saved new best model (mIoU=0.2636) ***


  *** Saved new best model (mIoU=0.2680) ***



Epoch 5/50  (lr=1.83e-04)
  Train Loss: 0.8262  |  Val Loss: 0.7878
  Train mIoU: 0.3546  |  Val mIoU: 0.2513
  Train Acc:  0.9017  |  Val Acc:  0.8875
  Val Per-Class:
    Class                IoU     Prec   Recall
    -----------------------------------------
    No Damage         0.9016   0.9955   0.9053
    Minor             0.2307   0.2840   0.5516
    Major             0.0017   0.0017   0.5758
    Destroyed         0.0111   0.0113   0.4614
    Unclassified      0.1114   0.5995   0.1204


  *** Saved new best model (mIoU=0.3193) ***


  *** Saved new best model (mIoU=0.3467) ***



Epoch 10/50  (lr=1.96e-04)
  Train Loss: 0.7675  |  Val Loss: 0.7281
  Train mIoU: 0.4148  |  Val mIoU: 0.3368
  Train Acc:  0.9219  |  Val Acc:  0.9499
  Val Per-Class:
    Class                IoU     Prec   Recall
    -----------------------------------------
    No Damage         0.9516   0.9931   0.9579
    Minor             0.4588   0.5155   0.8068
    Major             0.0036   0.0037   0.1622
    Destroyed         0.0091   0.0095   0.1833
    Unclassified      0.2606   0.3693   0.4697


  *** Saved new best model (mIoU=0.3533) ***


RuntimeError: Caught MemoryError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Python311\Lib\site-packages\torch\utils\data\_utils\worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Python311\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Python311\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "c:\Dev\Damage_Assessment_on_xBD\src\augment.py", line 65, in __getitem__
    post = np.array(Image.open(os.path.join(self.img_dir, post_name))).astype(np.float32) / 255.0
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^~~~~~~

In [ ]:
plot_train_history(history)

In [ ]:
test_loss, test_metrics = test_evaluation(model, test_loader, criterion, device, SAVE_PATH, NUM_CLASSES, siamese=True)